In [ ]:
%load_ext autoreload
%autoreload 2

import os,gc

import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from dataset import FacesDataset
from augmentations import get_train_transform, get_test_transform
from train_gan_ebm import train_gan_ebm_full,build_gan_models, train_gan_with_epoch_callback
from gan_generate import (
    generate_samples, show_images, save_samples, compute_fid, compute_is
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32

# Load data
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# Add image paths
for df in [train_df, test_df]:
    df["image_path"] = df["id"].apply(
        lambda x: f"data/processed_64/face-{int(x)}.png"
    )

# Create datasets and dataloaders
train_loader = DataLoader(
    FacesDataset(train_df, transform=get_train_transform(64)), 
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    FacesDataset(test_df, transform=get_test_transform(64)), 
    batch_size=batch_size, shuffle=False
)

# Configuration
base_config = {
    "latent_dim": 128,
    "g_channels": [512, 256, 128, 64],
    "d_channels": [64, 128, 256, 512],
    "use_batchnorm": True,

    # Activations
    "g_activation": "relu",      
    "e_activation": "leakyrelu",

    # Dropout
    "g_dropout": 0.0,            
    "e_dropout": 0.0,

    # Optim / training balance
    "lr": 2e-4,
    "lr_G": 2e-4,                
    "lr_E": 1e-4,                
    "k_steps": 5,                
    "g_steps": 1,                
    "real_ratio": 0.5,           
    "gp_lambda": 10.0,
}

In [ ]:
# Helper function for evaluation
def evaluate_and_save(G, device, latent_dim, save_path, test_loader=None):
    """Evaluate GAN and save generated samples"""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    fid_score = compute_fid(G, test_loader or train_loader, device, latent_dim=latent_dim)
    is_score = compute_is(G, device, latent_dim=latent_dim)
    save_samples(G, device, save_path, latent_dim=latent_dim)
    
    return fid_score, is_score

In [ ]:
# Train base model
print("Training base GAN model...")
G, E, G_losses, E_losses = train_gan_ebm_full(
    config=base_config,
    device=device,
    train_loader=train_loader,
    epochs=30
)

# Evaluate
print("Evaluating model...")
fid_score, is_score = evaluate_and_save(
    G, device, 
    base_config["latent_dim"],
    "outputs/gan_images/base_model.png",
    test_loader
)
print(f"Base Model - FID: {fid_score:.2f}, IS: {is_score:.2f}")

In [ ]:
# Ablation Study - Run multiple experiments with train/eval metrics
ablation_results_dir = os.path.join("outputs", "gan_ebm", "ablation_results")
os.makedirs(ablation_results_dir, exist_ok=True)
ablation_results_csv = os.path.join(ablation_results_dir, "ablation_results_run1.csv")
ablation_results_json = os.path.join(ablation_results_dir, "ablation_results_run1.json")

experiments = [
    {"name": "base", "latent_dim": 128},
    {"name": "latent_64", "latent_dim": 64},
    {"name": "latent_256", "latent_dim": 256},
    {"name": "no_batchnorm", "use_batchnorm": False},
    {"name": "low_lr", "lr": 1e-4},
]

results = []
for exp in experiments:
    print(f"\n{'='*50}")
    print(f"Running: {exp['name']}")
    print('='*50)
    
    config = {**base_config, **exp}
    
    # Determine changed parameters for this experiment
    changed_params = []
    if "latent_dim" in exp:
        changed_params.append(f"latent_dim={exp['latent_dim']}")
    if "use_batchnorm" in exp:
        changed_params.append(f"batchnorm={exp['use_batchnorm']}")
    if "lr" in exp:
        changed_params.append(f"lr={exp['lr']}")
    changed_text = ", ".join(changed_params) if changed_params else "None"
    
    try:
        G, E, G_losses, E_losses = train_gan_ebm_full(
            config=config,
            device=device,
            train_loader=train_loader,
            epochs=30
        )
        
        # Save image path
        image_path = f"outputs/gan_images/{exp['name']}.png"
        
        # Compute metrics on test set (eval)
        fid_eval, is_eval = evaluate_and_save(G, device, config["latent_dim"], image_path, test_loader)
        
        # Compute metrics on train set
        fid_train = compute_fid(G, train_loader, device, latent_dim=config["latent_dim"])
        is_train = compute_is(G, device, latent_dim=config["latent_dim"])
        
        # Average loss
        avg_loss = (G_losses[-1] + E_losses[-1]) / 2
        
        print(f"✓ {exp['name']} - Train FID: {fid_train:.2f}, Eval FID: {fid_eval:.2f}, Train IS: {is_train:.2f}, Eval IS: {is_eval:.2f}")
        
        results.append({
            "experiment": exp['name'],
            "image_path": image_path,
            "loss": avg_loss,
            "train_fid": fid_train,
            "eval_fid": fid_eval,
            "train_is": is_train,
            "eval_is": is_eval,
            "changed_params": changed_text,
            "latent_dim": config["latent_dim"],
            "batchnorm": config["use_batchnorm"],
            "lr": config["lr"],
            "G_loss": G_losses[-1],
            "E_loss": E_losses[-1]
        })
    except Exception as e:
        print(f"✗ Failed: {exp['name']} → {e}")

# Display results
if results:
    ablation_table = pd.DataFrame(results)
    ablation_table.to_csv(ablation_results_csv, index=False)
    ablation_table.to_json(ablation_results_json, orient="records", indent=2)
    print(f"\nSaved ablation results to {ablation_results_csv}")
    print(f"Saved ablation results to {ablation_results_json}")
    print("\n" + "="*50)
    print("ABLATION STUDY RESULTS")
    print("="*50)
    print(ablation_table[["experiment", "loss", "train_fid", "eval_fid", "train_is", "eval_is", "changed_params"]].to_string(index=False))

In [ ]:
# Ablation Cards Visualization Function
def plot_ablation_cards(df, figsize=(24, 8), title_fontsize=22, metrics_fontsize=15):
    """Plot ablation cards with experiment names, images, metrics, and changes."""
    import os

    n = len(df)
    fig, axes = plt.subplots(2, n, figsize=figsize)

    if n == 1:
        axes = [[axes[0]], [axes[1]]]

    for i, row in df.reset_index(drop=True).iterrows():
        # Top row: experiment name + image
        axes[0, i].set_title(row["experiment"], fontsize=title_fontsize, pad=12, fontweight="bold")
        img_path = str(row["image_path"])

        try:
            if not os.path.exists(img_path):
                raise FileNotFoundError(img_path)
            img = plt.imread(img_path)
            axes[0, i].imshow(img)
        except Exception:
            axes[0, i].text(0.5, 0.5, "Image not found", ha="center", va="center", fontsize=metrics_fontsize)
        axes[0, i].axis("off")

        # Bottom row: metrics + changed params
        axes[1, i].axis("off")
        text = (
            f"Loss: {row['loss']:.4f}\n\n"
            f"Train FID: {row['train_fid']:.4f}\n\n"
            f"Train IS: {row['train_is']:.4f}\n\n"
            f"Eval FID: {row['eval_fid']:.4f}\n\n"
            f"Eval IS: {row['eval_is']:.4f}\n\n"
            f"Changes: {row['changed_params']}"
        )
        axes[1, i].text(
            0.5,
            0.5,
            text,
            ha="center",
            va="center",
            fontsize=metrics_fontsize,
            linespacing=1.6,
        )

    plt.subplots_adjust(hspace=0.4, wspace=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Display Ablation Cards
plot_ablation_cards(ablation_table)

In [ ]:
# Ablation Study - Run multiple experiments with train/eval metrics
ablation_results_dir = os.path.join("outputs", "gan_ebm", "ablation_results")
os.makedirs(ablation_results_dir, exist_ok=True)
ablation_results_csv = os.path.join(ablation_results_dir, "ablation_results_run2.csv")
ablation_results_json = os.path.join(ablation_results_dir, "ablation_results_run2.json")

experiments = [
    {"name": "base"},
    {"name": "k3_g1", "k_steps": 3, "g_steps": 1},
    {"name": "k5_g2", "k_steps": 5, "g_steps": 2},
    {"name": "g_relu_e_relu", "g_activation": "relu", "e_activation": "relu"},
    {"name": "g_gelu_e_leaky", "g_activation": "gelu", "e_activation": "leakyrelu"},
    {"name": "real70_fake30", "real_ratio": 0.7},
    {"name": "real30_fake70", "real_ratio": 0.3},
    {"name": "dropout_10", "g_dropout": 0.1, "e_dropout": 0.1},
    {"name": "dropout_20", "g_dropout": 0.2, "e_dropout": 0.2},
]

results = []
for exp in experiments:
    print(f"\n{'='*50}")
    print(f"Running: {exp['name']}")
    print('='*50)
    
    config = {**base_config, **exp}
    
    # Determine changed parameters for this experiment
    changed_params = []
    if "k_steps" in exp:
        changed_params.append(f"k_steps={exp['k_steps']}")
    if "g_steps" in exp:
        changed_params.append(f"g_steps={exp['g_steps']}")
    if "g_activation" in exp:
        changed_params.append(f"g_activation={exp['g_activation']}")
    if "e_activation" in exp:
        changed_params.append(f"e_activation={exp['e_activation']}")
    if "real_ratio" in exp:
        changed_params.append(f"real_ratio={exp['real_ratio']}")
    if "g_dropout" in exp:
        changed_params.append(f"g_dropout={exp['g_dropout']}")
    if "e_dropout" in exp:
        changed_params.append(f"e_dropout={exp['e_dropout']}")
    if "lr" in exp:
        changed_params.append(f"lr={exp['lr']}")
    changed_text = ", ".join(changed_params) if changed_params else "None"
    
    try:
        G, E, G_losses, E_losses = train_gan_ebm_full(
            config=config,
            device=device,
            train_loader=train_loader,
            epochs=5
        )
        
        # Save image path
        image_path = f"outputs/gan_images/{exp['name']}.png"
        
        # Compute metrics on test set (eval)
        fid_eval, is_eval = evaluate_and_save(G, device, config["latent_dim"], image_path, test_loader)
        
        # Compute metrics on train set
        fid_train = compute_fid(G, train_loader, device, latent_dim=config["latent_dim"])
        is_train = compute_is(G, device, latent_dim=config["latent_dim"])
        
        # Average loss
        avg_loss = (G_losses[-1] + E_losses[-1]) / 2
        
        print(f"✓ {exp['name']} - Train FID: {fid_train:.2f}, Eval FID: {fid_eval:.2f}, Train IS: {is_train:.2f}, Eval IS: {is_eval:.2f}")
        
        results.append({
            "experiment": exp['name'],
            "image_path": image_path,
            "loss": avg_loss,
            "train_fid": fid_train,
            "eval_fid": fid_eval,
            "train_is": is_train,
            "eval_is": is_eval,
            "changed_params": changed_text,
            "latent_dim": config["latent_dim"],
            "batchnorm": config["use_batchnorm"],
            "lr": config["lr"],
            "k_steps": config["k_steps"],
            "g_steps": config["g_steps"],
            "g_activation": config["g_activation"],
            "e_activation": config["e_activation"],
            "real_ratio": config["real_ratio"],
            "g_dropout": config["g_dropout"],
            "e_dropout": config["e_dropout"],
            "G_loss": G_losses[-1],
            "E_loss": E_losses[-1]
        })
    except Exception as e:
        print(f"✗ Failed: {exp['name']} → {e}")

# Display results
if results:
    ablation_table = pd.DataFrame(results)
    ablation_table.to_csv(ablation_results_csv, index=False)
    ablation_table.to_json(ablation_results_json, orient="records", indent=2)
    print(f"\nSaved ablation results to {ablation_results_csv}")
    print(f"Saved ablation results to {ablation_results_json}")
    print("\n" + "="*50)
    print("ABLATION STUDY RESULTS")
    print("="*50)
    print(ablation_table[["experiment", "loss", "train_fid", "eval_fid", "train_is", "eval_is", "changed_params"]].to_string(index=False))

# Keep a live table available in the notebook session
ablation_table = pd.DataFrame(results) if results else pd.DataFrame()

In [ ]:
# Load saved ablation results for reformatting or reuse
ablation_result_run = 1  # Change to 2 to load the second saved ablation table
ablation_results_dir = os.path.join("outputs", "gan_ebm", "ablation_results")
ablation_results_csv = os.path.join(ablation_results_dir, f"ablation_results_run{ablation_result_run}.csv")
ablation_results_json = os.path.join(ablation_results_dir, f"ablation_results_run{ablation_result_run}.json")

if os.path.exists(ablation_results_csv):
    ablation_table = pd.read_csv(ablation_results_csv)
    print(f"Loaded ablation results from {ablation_results_csv}")
elif os.path.exists(ablation_results_json):
    ablation_table = pd.read_json(ablation_results_json)
    print(f"Loaded ablation results from {ablation_results_json}")
else:
    if "ablation_table" in globals() and not ablation_table.empty:
        print("Using in-memory ablation_table because no saved results file was found.")
    else:
        raise FileNotFoundError(
            f"No saved ablation results found for run {ablation_result_run}. Run the matching ablation cell first.",
        )

# Re-display table from saved results so formatting can be changed without rerunning experiments
if not ablation_table.empty:
    print("\n" + "="*50)
    print("ABLATION STUDY RESULTS")
    print("="*50)
    display(ablation_table[["experiment", "loss", "train_fid", "eval_fid", "train_is", "eval_is", "changed_params"]])

In [ ]:
# Train final model with checkpoints, image generation, and metrics evaluation
print("Training final model (300 epochs with checkpoints and evaluation)...")

EPOCHS = 300
CHECKPOINT_INTERVAL = 20  # Save every 20 epochs
EVAL_INTERVAL = 5  # Evaluate every 5 epochs

checkpoint_dir = "models/gan_checkpoints"
image_checkpoint_dir = "outputs/gan_checkpoints"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(image_checkpoint_dir, exist_ok=True)

metrics_history = {
    "epoch": [],
    "FID": [],
    "G_loss": [],
    "E_loss": [],
}

# Create generator and discriminator for final training
G_final, D_final = build_gan_models(final_config, device)

def checkpoint_callback(epoch, generator, discriminator, g_loss, d_loss):
    """Called after each epoch to save checkpoints and evaluate."""
    epoch_num = epoch + 1
    # Save checkpoint every 20 epochs
    if epoch_num % CHECKPOINT_INTERVAL == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"gan_epoch_{epoch_num}.pth")
        torch.save(
            {
                "generator_state_dict": generator.state_dict(),
                "discriminator_state_dict": discriminator.state_dict(),
                "epoch": epoch_num,
                "config": final_config,
            },
            checkpoint_path,
        )
        print(f"Checkpoint saved: epoch {epoch_num}")

        # Generate images from this checkpoint
        sample_path = os.path.join(image_checkpoint_dir, f"gan_epoch_{epoch_num}.png")
        save_samples(generator, device, sample_path, latent_dim=final_config["latent_dim"])
        print(f"Generated images saved: {sample_path}")

    # Evaluate every 5 epochs
    if epoch_num % EVAL_INTERVAL == 0:
        print(f"\nEvaluating at epoch {epoch_num}...")
        fid_score = compute_fid(
            generator,
            test_loader,
            device,
            latent_dim=final_config["latent_dim"],
        )

        metrics_history["epoch"].append(epoch_num)
        metrics_history["FID"].append(fid_score)
        metrics_history["G_loss"].append(g_loss)
        metrics_history["E_loss"].append(d_loss)

        print(f"Epoch {epoch_num} - FID: {fid_score:.2f}\n")

    # Clear VRAM for the next epoch/model step
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

G_losses_final, E_losses_final = train_gan_with_epoch_callback(
    generator=G_final,
    discriminator=D_final,
    dataloader=train_loader,
    device=device,
    epochs=EPOCHS,
    latent_dim=final_config["latent_dim"],
    lr=final_config["lr"],
    epoch_callback=checkpoint_callback,
)

# Save final checkpoint in the same format as periodic checkpoints
final_ckpt_model_path = os.path.join(checkpoint_dir, "gan_final.pth")
final_ckpt_output_path = "outputs/final_generator.pth"

final_payload = {
    "generator_state_dict": G_final.state_dict(),
    "discriminator_state_dict": D_final.state_dict(),
    "epoch": EPOCHS,
    "config": final_config,
}

torch.save(final_payload, final_ckpt_model_path)
torch.save(final_payload, final_ckpt_output_path)

print(f"✓ Final checkpoint saved to {final_ckpt_model_path}")
print(f"✓ Final checkpoint saved to {final_ckpt_output_path}")
print("\n✓ Training completed!")

In [ ]:
# Visualize metrics against epochs
if metrics_history["epoch"]:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # FID Score Plot
    axes[0, 0].plot(metrics_history["epoch"], metrics_history["FID"], marker='o', linewidth=2, markersize=6)
    axes[0, 0].set_xlabel("Epoch", fontsize=12)
    axes[0, 0].set_ylabel("FID Score", fontsize=12)
    axes[0, 0].set_title("FID Score vs Epoch", fontsize=14, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Generator Loss at Evaluation Epochs
    axes[0, 1].plot(metrics_history["epoch"], metrics_history["G_loss"], marker='o', color='orange', linewidth=2, markersize=6)
    axes[0, 1].set_xlabel("Epoch", fontsize=12)
    axes[0, 1].set_ylabel("Generator Loss", fontsize=12)
    axes[0, 1].set_title("Generator Loss at Eval Epochs", fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Generator Loss Plot
    axes[1, 0].plot(range(1, len(G_losses_final) + 1), G_losses_final, label='Generator Loss', linewidth=2, color='green')
    axes[1, 0].set_xlabel("Epoch", fontsize=12)
    axes[1, 0].set_ylabel("Loss", fontsize=12)
    axes[1, 0].set_title("Generator Loss vs Epoch", fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    # Discriminator Loss Plot
    axes[1, 1].plot(range(1, len(E_losses_final) + 1), E_losses_final, label='Discriminator Loss', linewidth=2, color='red')
    axes[1, 1].set_xlabel("Epoch", fontsize=12)
    axes[1, 1].set_ylabel("Loss", fontsize=12)
    axes[1, 1].set_title("Discriminator Loss vs Epoch", fontsize=14, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig("outputs/gan_metrics_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print metrics summary
    print("\n" + "="*60)
    print("METRICS SUMMARY")
    print("="*60)
    metrics_df = pd.DataFrame(metrics_history)
    print(metrics_df.to_string(index=False))
    print("="*60)
else:
    print("No metrics recorded. Ensure evaluation was run during training.")

In [ ]:
# Load and inference from saved checkpoint
from generator import Generator

LOAD_EPOCH = 300  

checkpoint_dir = "models/gan_checkpoints"

if LOAD_EPOCH is not None:
    # Load specific epoch checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, f"gan_epoch_{LOAD_EPOCH}.pth")
else:
    # Load final checkpoint
    checkpoint_path = os.path.join(checkpoint_dir, "gan_final.pth")

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found at: {checkpoint_path}")

print(f"Loading saved model from: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=device)
config_loaded = checkpoint["config"]

G_loaded = Generator(
    latent_dim=config_loaded["latent_dim"],
    channels=config_loaded["g_channels"],
    use_batchnorm=config_loaded["use_batchnorm"],
    activation=config_loaded["activation"]
).to(device)

G_loaded.load_state_dict(checkpoint["generator_state_dict"])

# Generate samples from loaded model
print("Generating samples from loaded model...")
samples = generate_samples(G_loaded, device, config_loaded["latent_dim"], n_samples=16)
show_images(samples, "Samples from Loaded Model")

# Evaluate loaded model
fid_loaded, is_loaded = evaluate_and_save(
    G_loaded, device,
    config_loaded["latent_dim"],
    "outputs/loaded_model_samples.png",
    test_loader
)
print(f"Loaded Model - FID: {fid_loaded:.2f}, IS: {is_loaded:.2f}")